In [20]:

building = [
    [1, 1, 0, 1],
    [0, 1, 1, 1],
    [1, 1, 0, 1],
    [1, 0, 1, 1]
]

def grid_to_graph(grid):
    rows, cols = len(grid), len(grid[0])
    graph = {}
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 1:
                neighbors = []
                for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nr, nc = r+dr, c+dc
                    if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 1:
                        neighbors.append((nr, nc))
                graph[(r, c)] = neighbors
    return graph

class GoalBasedAgent:
    def __init__(self, goal):
        self.goal = goal

    def formulate_goal(self, percept):
        if percept == self.goal:
            return "Goal reached"
        return "Searching"

    def bfs_search(self, graph, start, goal):
        visited = []
        queue = []
        visited.append(start)
        queue.append(start)
        traversal_order = []

        while queue:
            node = queue.pop(0)
            traversal_order.append(node)
            print(f"  Visiting: {node}")
            if node == goal:
                return traversal_order, f"Goal {goal} found!"
            for neighbour in graph.get(node, []):
                if neighbour not in visited:
                    visited.append(neighbour)
                    queue.append(neighbour)
        return traversal_order, "Goal not found"

    def act(self, percept, graph):
        goal_status = self.formulate_goal(percept)
        if goal_status == "Goal reached":
            return [], f"Goal {self.goal} found at start!"
        else:
            return self.bfs_search(graph, percept, self.goal)

class Environment:
    def __init__(self, graph):
        self.graph = graph

    def get_percept(self, node):
        return node

def bfs_shortest_path(graph, start, goal):
    queue = deque([[start]])
    visited = set([start])
    while queue:
        path = queue.popleft()
        node = path[-1]
        if node == goal:
            return path
        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(path + [neighbor])
    return None


graph = grid_to_graph(building)
agent   = GoalBasedAgent((3, 3))
env     = Environment(graph)
percept = env.get_percept((0, 0))
traversal, msg = agent.act(percept, graph)
print(f"\nTraversal Order: {traversal}")
path = bfs_shortest_path(graph, (0, 0), (3, 3))
print(f"Shortest Path  : {path}")
print(msg)

  Visiting: (0, 0)
  Visiting: (0, 1)
  Visiting: (1, 1)
  Visiting: (2, 1)
  Visiting: (1, 2)
  Visiting: (2, 0)
  Visiting: (1, 3)
  Visiting: (3, 0)
  Visiting: (0, 3)
  Visiting: (2, 3)
  Visiting: (3, 3)

Traversal Order: [(0, 0), (0, 1), (1, 1), (2, 1), (1, 2), (2, 0), (1, 3), (3, 0), (0, 3), (2, 3), (3, 3)]
Shortest Path  : [(0, 0), (0, 1), (1, 1), (1, 2), (1, 3), (2, 3), (3, 3)]
Goal (3, 3) found!


In [21]:


graph = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F'],
    'D': ['G'],
    'E': [],
    'F': ['H'],
    'G': [],
    'H': []
}

def dls(graph, node, goal, depth, path, visited):
    print(f"  Visiting: {node} (depth={len(path)-1})")
    if node == goal:
        return path
    if depth == 0:
        return None
    for neighbor in graph.get(node, []):
        if neighbor not in visited:
            visited.add(neighbor)
            result = dls(graph, neighbor, goal, depth - 1, path + [neighbor], visited)
            if result:
                return result
            visited.discard(neighbor)
    return None


for limit in [2, 3]:
    print(f"\n Depth Limit = {limit} ")
    result = dls(graph, 'A', 'H', limit, ['A'], {'A'})
    if result:
        print("Path found:", result)
    else:
        print(f"Goal not found within depth {limit}")



 Depth Limit = 2 
  Visiting: A (depth=0)
  Visiting: B (depth=1)
  Visiting: D (depth=2)
  Visiting: E (depth=2)
  Visiting: C (depth=1)
  Visiting: F (depth=2)
Goal not found within depth 2

 Depth Limit = 3 
  Visiting: A (depth=0)
  Visiting: B (depth=1)
  Visiting: D (depth=2)
  Visiting: G (depth=3)
  Visiting: E (depth=2)
  Visiting: C (depth=1)
  Visiting: F (depth=2)
  Visiting: H (depth=3)
Path found: ['A', 'C', 'F', 'H']


In [ ]:


graph = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F'],
    'D': ['G'],
    'E': [],
    'F': ['H'],
    'G': [],
    'H': []
}


def dls_ids(graph, node, goal, depth, path):
    if node == goal:
        return path
    if depth == 0:
        return None
    for neighbor in graph.get(node, []):
        if neighbor not in path:
            result = dls_ids(graph, neighbor, goal, depth - 1, path + [neighbor])
            if result:
                return result
    return None

def ids(graph, start, goal):
    depth = 0
    while True:
        print(f"\n  Depth Level {depth}:")
        result = dls_ids(graph, start, goal, depth, [start])
        if result:
            print(f"  Goal found! Path: {result}")
            return result
        print("  Not found at this depth.")
        depth += 1


ids(graph, 'A', 'G')


  Depth Level 0:
  Not found at this depth.

  Depth Level 1:
  Not found at this depth.

  Depth Level 2:
  Not found at this depth.

  Depth Level 3:
  Goal found! Path: ['A', 'B', 'D', 'G']


['A', 'B', 'D', 'G']

In [28]:

import heapq
graph = {
    'S': {'A': 4, 'B': 2},
    'A': {'C': 5, 'D': 10},
    'B': {'E': 3},
    'C': {'G': 4},
    'D': {'G': 1},
    'E': {'D': 4},
    'G': {}
}

def ucs(graph, start, goal):
    frontier = [(0, start, [start])]
    visited = {}
    while frontier:
        cost, node, path = heapq.heappop(frontier)
        if node in visited and visited[node] <= cost:
            continue
        visited[node] = cost
        print(f"  Exploring: {node} , cost={cost}")
        if node == goal:
            return path, cost
        for neighbor, w in graph.get(node, {}).items():
            new_cost = cost + w
            heapq.heappush(frontier, (new_cost, neighbor, path + [neighbor]))
    return None, float('inf')


path, cost = ucs(graph, 'S', 'G')
print(f"\nLeast-Cost Path: {path}")
print(f"Total Cost     : {cost}")

  Exploring: S , cost=0
  Exploring: B , cost=2
  Exploring: A , cost=4
  Exploring: E , cost=5
  Exploring: C , cost=9
  Exploring: D , cost=9
  Exploring: G , cost=10

Least-Cost Path: ['S', 'B', 'E', 'D', 'G']
Total Cost     : 10


In [26]:
from collections import deque
from queue import PriorityQueue

class Node:
    def __init__(self, position, parent=None):
        self.position = position
        self.parent   = parent
        self.g = 0   # cost from start to current node
        self.h = 0   # heuristic estimate to end node
        self.f = 0   # total cost

    def __lt__(self, other):
        return self.f < other.f

def heuristic(current_pos, end_pos):
    return abs(current_pos[0] - end_pos[0]) + abs(current_pos[1] - end_pos[1])

def best_first_search(maze, start, end):
    rows, cols = len(maze), len(maze[0])
    start_node = Node(start)
    end_node   = Node(end)
    frontier   = PriorityQueue()
    frontier.put(start_node)
    visited    = set()

    while not frontier.empty():
        current_node = frontier.get()
        current_pos  = current_node.position

        if current_pos == end:
            path = []
            while current_node:
                path.append(current_node.position)
                current_node = current_node.parent
            return path[::-1]   
        visited.add(current_pos)

        for dx, dy in [(1,0),(-1,0),(0,1),(0,-1)]:
            new_pos = (current_pos[0] + dx, current_pos[1] + dy)
            if (0 <= new_pos[0] < rows and 0 <= new_pos[1] < cols
                    and maze[new_pos[0]][new_pos[1]] == 0
                    and new_pos not in visited):
                new_node   = Node(new_pos, current_node)
                new_node.g = current_node.g + 1
                new_node.h = heuristic(new_pos, end)
                new_node.f = new_node.h  
                frontier.put(new_node)
                visited.add(new_pos)

    return None  
graph = {
    'A': {'B': 2, 'C': 1},
    'B': {'D': 4, 'E': 3},
    'C': {'F': 1, 'G': 5},
    'D': {'H': 2},
    'E': {},
    'F': {'I': 6},
    'G': {},
    'H': {},
    'I': {}
}

heuristic_task5 = {'A': 7, 'B': 6, 'C': 5, 'D': 4, 'E': 3, 'F': 3, 'G': 6, 'H': 2, 'I': 0}

def greedy_bfs(graph, start, goal):
    frontier  = [[start, heuristic_task5[start]]]   
    visited   = set()
    came_from = {start: None}

    while frontier:
        frontier.sort(key=lambda x: x[1])
        current_node, _ = frontier.pop(0)

        if current_node in visited:
            continue

        print(current_node, end=" ")
        visited.add(current_node)

        if current_node == goal:
            path = []
            while current_node is not None:
                path.append(current_node)
                current_node = came_from[current_node]
            path.reverse()
            print(f"\nGoal found with GBFS. Path: {path}")
            return path

        for neighbor in graph[current_node]:
            if neighbor not in visited:
                came_from[neighbor] = current_node
                frontier.append([neighbor, heuristic_task5[neighbor]])

    print("\nGoal not found")
    return None


# Part A: Grid-based Best-First Search
maze_grid = [
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [1, 0, 0, 0],
    [0, 0, 1, 0]
]
start = (0, 0)
end   = (3, 3)
print("\nPart A: Grid Maze — Best-First Search:")
path = best_first_search(maze_grid, start, end)
if path:
    print("Path found:", path)
else:
    print("No path found")

# Part B: Graph-based Greedy Best-First Search 
print("\nPart B: Graph — Greedy Best-First Search:")
print("Following is the Greedy Best-First Search (GBFS):")
greedy_bfs(graph, 'A', 'I')



Part A: Grid Maze — Best-First Search:
Path found: [(0, 0), (1, 0), (1, 1), (2, 1), (2, 2), (2, 3), (3, 3)]

Part B: Graph — Greedy Best-First Search:
Following is the Greedy Best-First Search (GBFS):
A C F I 
Goal found with GBFS. Path: ['A', 'C', 'F', 'I']


['A', 'C', 'F', 'I']

In [27]:

import random, copy
graph = {
    'S': {'A': 4, 'B': 2},
    'A': {'C': 5, 'D': 10},
    'B': {'E': 3},
    'C': {'G': 4},
    'D': {'G': 1},
    'E': {'D': 4},
    'G': {}
}

heuristic= {'S': 7, 'A': 5, 'B': 6, 'C': 3, 'D': 2, 'E': 4, 'G': 0}

# A* using manual sorted frontier (matching image code style)
def a_star(graph, start, goal):
    visited   = set()
    g_costs   = {start: 0}
    came_from = {start: None}
    frontier  = [[start, heuristic[start]]]   # [node, f = g + h]

    while frontier:
        # Sort frontier by f(n) = g(n) + h(n)
        frontier.sort(key=lambda x: x[1])
        current_node, current_f = frontier.pop(0)

        if current_node in visited:
            continue

        print(current_node, end=" ")
        visited.add(current_node)

        if current_node == goal:
            path = []
            while current_node is not None:
                path.append(current_node)
                current_node = came_from[current_node]
            path.reverse()
            print(f"\nGoal found with A*. Path: {path}")
            return path, g_costs[goal]

        for neighbor, cost in graph[current_node].items():
            new_g  = g_costs[current_node] + cost         
            f_cost = new_g + heuristic[neighbor]    # f(n) = g(n) + h(n)

            if neighbor not in g_costs or new_g < g_costs[neighbor]:
                g_costs[neighbor]   = new_g
                came_from[neighbor] = current_node
                frontier.append([neighbor, f_cost])

    print("\nGoal not found")
    return None, float('inf')

def dynamic_edge_change(graph):
    edges = [(u, v) for u, nbrs in graph.items() for v in nbrs if nbrs]
    u, v  = random.choice(edges)
    old   = graph[u][v]
    delta = random.choice([-3, -2, 2, 3, 4])
    new   = max(1, old + delta)
    graph[u][v] = new
    print(f"  Edge {u}→{v} changed: {old} → {new}")
    return graph


random.seed(42)
graph = copy.deepcopy(graph)
for iteration in range(1, 4):
    print(f"\n Iteration {iteration}")
    if iteration > 1:
        graph = dynamic_edge_change(graph)
    path, cost = a_star(graph, 'S', 'G')
    print(f"  Optimal Path: {path}")
    print(f"  Total Cost  : {cost}")


 Iteration 1
S B A E D G 
Goal found with A*. Path: ['S', 'B', 'E', 'D', 'G']
  Optimal Path: ['S', 'B', 'E', 'D', 'G']
  Total Cost  : 10

 Iteration 2
  Edge S→B changed: 2 → 1
S B E A D G 
Goal found with A*. Path: ['S', 'B', 'E', 'D', 'G']
  Optimal Path: ['S', 'B', 'E', 'D', 'G']
  Total Cost  : 9

 Iteration 3
  Edge B→E changed: 3 → 1
S B E D G 
Goal found with A*. Path: ['S', 'B', 'E', 'D', 'G']
  Optimal Path: ['S', 'B', 'E', 'D', 'G']
  Total Cost  : 7
